In [ ]:
import pandas as pd
import os
from highstreets import config
from highstreets.data_source_sink.dataloader import DataLoader
from highstreets.data_source_sink.datawriter import DataWriter
from highstreets.data_transformation.mcard_transform import McardTransform
from highstreets.core.sql_manager import SQLManager
from highstreets.data_transformation.mcard_weekly_processor import FileProcessor
from highstreets.api.clientbase import APIClient
from sqlalchemy import create_engine
import psycopg2
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

base_dir = config.BASE_DIR
# initialize the database connection
database = os.getenv("PG_DATABASE")
username = os.getenv("PG_USER")
password = os.getenv("PG_PASSWORD")
host = os.getenv("PG_HOST")
port = os.getenv("PG_PORT")
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@" f"{host}:{port}/{database}"
)

# instantiate the classes
data_loader = DataLoader()
data_writer = DataWriter()
mcard_transform = McardTransform()
sql_manager = SQLManager()
api_client = APIClient()
dir_path = f"{base_dir}mastercard/sharefile_test"
mcard_weekly = FileProcessor(data_loader, data_writer, dir_path)

# Connect to PostgreSQL database
conn = psycopg2.connect(
    dbname=os.getenv("PG_DATABASE"),
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"),
    port=os.getenv("PG_PORT"),
)

In [ ]:
print(base_dir)

In [ ]:
txn_tc = pd.read_csv("Z:/HSDS/data/mastercard/weekly/processed/test/adjusted_weekly_data/txn_towncentre.csv")
txn_tc_ds = pd.read_csv("Z:/HSDS/data/mastercard/weekly/processed/txn_towncentres (1).csv")
txn_tc_feb = pd.read_csv("Z:/HSDS/data/mastercard/weekly/processed/txn_towncentres.csv")

txn_tc['week_start'] = pd.to_datetime(txn_tc['week_start'])
txn_tc_ds['week_start'] = pd.to_datetime(txn_tc_ds['week_start'])
txn_tc_feb['week_start'] = pd.to_datetime(txn_tc_feb['week_start'])

In [ ]:
txn_tc[(txn_tc['tc_name']=='West End')][['yr','wk','week_start','txn_amt_wd_eating','txn_amt_wd_eating_adj']].tail()

In [ ]:
adj = data_loader.get_full_data("econ_busyness_mcard_adjustment_factors")
adj[(adj['inner_outer']=='Inner')].tail(5)

In [ ]:
cpi = data_loader.get_full_data("econ_busyness_mcard_cpi_data")

def reindex_cpi(x,yr=2018):
    reindex = x[x["yr"] == yr]["cpi_index"].mean()
    x["cpi_index"] = x["cpi_index"] / reindex * 100
    return x

cpi = cpi.groupby(['aggregate'],group_keys=False).apply(lambda x: reindex_cpi(x,yr=2018)).round(4)

cpi[(cpi['aggregate']=='Overall Index')|(cpi['aggregate'].str.contains("Restaurants|Clothing"))].sort_values(by=['yr','month','aggregate']).tail(10)

In [ ]:
426557.97/136.7054*100/1.652

In [ ]:
lookup = data_loader.get_full_data("econ_busyness_mcard_bespoke_quad_lookup")
unique_quads = lookup[lookup['name'].str.contains('Royal Hill')].sort_values(by=['name','quad_id'])['quad_id'].unique()
rh_lookup = lookup[lookup['name'].str.contains('Royal Hill')].sort_values(by=['name','quad_id'])

In [ ]:
rh_lookup

In [ ]:
rh_lookup['name'].value_counts()

In [ ]:
royal_hill_quads = data_loader.get_partial_data("gisapdata.econ_busyness_mcard_stg_18_zoom",columns='*'
                             ,where_clause=f"quad_id IN{tuple(list(unique_quads))} AND segment='Overall'")

In [ ]:
rh_lookup['quad_id'] = rh_lookup['quad_id'].astype('Int64')
royal_hill_quads['quad_id'] = royal_hill_quads['quad_id'].astype('Int64')

royal_hill = pd.merge(royal_hill_quads,rh_lookup,on='quad_id',how='inner')
royal_hill

In [ ]:
royal_hill = royal_hill.groupby(['yr','wk','industry','weekday_weekend','bespoke_area_id','name'])['txn_amt'].sum(min_count=1).reset_index()
royal_hill

In [ ]:
txn_bespoke = pd.read_csv("Z:/HSDS/data/mastercard/weekly/processed/test/adjusted_weekly_data/txn_bespoke.csv")
txn_bespoke['week_start'] = pd.to_datetime(txn_bespoke['week_start'])
txn_bespoke[txn_bespoke['name'].str.contains('Royal Hill')]


In [ ]:
royal_hill_quads

In [ ]:

import matplotlib.pyplot as plt

#df = merged_af_io[merged_af_io['inner_outer']==io]
df_new_sp = royal_hill_quads[(
    royal_hill_quads['quad_id'].isin(
    rh_lookup[rh_lookup['name']=='Royal Hill - spend']['quad_id'].unique()))&(
        royal_hill_quads['weekday_weekend']=='weekdays')&(
            royal_hill_quads['industry']=='Total Retail')]

#df1 = adj_factor_new[adj_factor_new['inner_outer']==io]
plt.figure(figsize=(14,4))

for quad in df_new_sp['quad_id'].unique():
    single = df_new_sp[df_new_sp['quad_id']==quad]
# This one is currently wrong:
#plt.plot(df['count_date'],df['txn_amt_adj'],label='txn_amt_adj') 
#plt.plot(df['count_date'],df['txn_amt_adj_new'],label='Correct version - adjusted in Python')
#plt.plot(df_new['week_start'],df_new['txn_amt'],label='txn_amt')
    plt.plot(single['txn_amt'],label=quad)

#plt.plot(df_feb['week_start'],df_feb['txn_amt_wd_retail_adj'],label='Feb')

plt.legend()   #plt.plot(df1['adjustment_factor_retail'])
plt.ylim(ymin=0)
#plt.xlim(xmin=np.datetime64('2020-10-01'),xmax=np.datetime64('2021-10-01'))
plt.title('Royal Hill - spend quads')
plt.show()

In [ ]:
df_new_sp = royal_hill_quads[(
    royal_hill_quads['quad_id'].isin(
    rh_lookup[rh_lookup['name']=='Royal Hill - footfall']['quad_id'].unique()))&(
        royal_hill_quads['weekday_weekend']=='weekdays')&(
            royal_hill_quads['industry']=='Total Retail')]

#df1 = adj_factor_new[adj_factor_new['inner_outer']==io]
plt.figure(figsize=(14,4))

for quad in df_new_sp['quad_id'].unique():
    single = df_new_sp[df_new_sp['quad_id']==quad]
# This one is currently wrong:
#plt.plot(df['count_date'],df['txn_amt_adj'],label='txn_amt_adj') 
#plt.plot(df['count_date'],df['txn_amt_adj_new'],label='Correct version - adjusted in Python')
#plt.plot(df_new['week_start'],df_new['txn_amt'],label='txn_amt')
    plt.plot(single['txn_amt'],label=quad)

#plt.plot(df_feb['week_start'],df_feb['txn_amt_wd_retail_adj'],label='Feb')

plt.legend()   #plt.plot(df1['adjustment_factor_retail'])
plt.ylim(ymin=0)
#plt.xlim(xmin=np.datetime64('2020-10-01'),xmax=np.datetime64('2021-10-01'))
plt.title('Royal Hill - footfall quads')
plt.show()

In [ ]:
import matplotlib.pyplot as plt


#df = merged_af_io[merged_af_io['inner_outer']==io]
df_new = txn_bespoke[txn_bespoke['name']=='Royal Hill']
df_new_sp = txn_bespoke[txn_bespoke['name']=='Royal Hill - spend']
df_new_foot = txn_bespoke[txn_bespoke['name']=='Royal Hill - footfall']

#df1 = adj_factor_new[adj_factor_new['inner_outer']==io]
plt.figure(figsize=(14,4))
# This one is currently wrong:
#plt.plot(df['count_date'],df['txn_amt_adj'],label='txn_amt_adj') 
#plt.plot(df['count_date'],df['txn_amt_adj_new'],label='Correct version - adjusted in Python')
#plt.plot(df_new['week_start'],df_new['txn_amt'],label='txn_amt')
plt.plot(df_new['week_start'],df_new['txn_amt_wd_retail_adj'],label='Royal Hill')
plt.plot(df_new_sp['week_start'],df_new_sp['txn_amt_wd_retail_adj'],label='Royal Hill - spend')
plt.plot(df_new_foot['week_start'],df_new_foot['txn_amt_wd_retail_adj'],label='Royal Hill - footfall')

#plt.plot(df_feb['week_start'],df_feb['txn_amt_wd_retail_adj'],label='Feb')

plt.legend()   #plt.plot(df1['adjustment_factor_retail'])
plt.ylim(ymin=0)
#plt.xlim(xmin=np.datetime64('2020-10-01'),xmax=np.datetime64('2021-10-01'))
plt.title('Royal Hill')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

## Plot these differences between txn_amt_adj and txn_amt_adj_new 
for name in txn_tc['tc_name'].unique()[:5]:
    #df = merged_af_io[merged_af_io['inner_outer']==io]
    df_new = txn_tc[txn_tc['tc_name']==name]
    df_ds = txn_tc_ds[txn_tc_ds['tc_name']==name]
    df_feb = txn_tc_feb[txn_tc_feb['tc_name']==name]

    #df1 = adj_factor_new[adj_factor_new['inner_outer']==io]
    plt.figure(figsize=(14,4))
    # This one is currently wrong:
    #plt.plot(df['count_date'],df['txn_amt_adj'],label='txn_amt_adj') 
    #plt.plot(df['count_date'],df['txn_amt_adj_new'],label='Correct version - adjusted in Python')
    #plt.plot(df_new['week_start'],df_new['txn_amt'],label='txn_amt')
    plt.plot(df_new['week_start'],df_new['txn_amt_wd_retail'],label='Unadjusted')

    plt.plot(df_new['week_start'],df_new['txn_amt_wd_retail_adj'],label='New')
    plt.plot(df_ds['week_start'],df_ds['txn_amt_wd_retail_adj'],label='DS')
    #plt.plot(df_feb['week_start'],df_feb['txn_amt_wd_retail_adj'],label='Feb')

    plt.legend()   #plt.plot(df1['adjustment_factor_retail'])
    plt.ylim(ymin=0)
    #plt.xlim(xmin=np.datetime64('2020-10-01'),xmax=np.datetime64('2021-10-01'))
    plt.title(name)
    plt.show()